# 🚀 Model Deployment - Champion Promotion

## Purpose
Promotes approved Challenger model to Champion and creates serving endpoint

## Steps
1. Verify Challenger model approval
2. Promote Challenger to Champion alias
3. Create/update Model Serving endpoint
4. Test endpoint with sample data
5. Enable endpoint for production inference

## Notes
- Final step of 3-step deployment workflow (Evaluation → Approval → **Deployment**)
- Follows Iris MLOps pattern for production deployment
- Creates real-time inference endpoint for water quality predictions
- Champion model becomes the production model for batch and real-time inference

In [ ]:
# 📦 Install required packages
%pip install mlflow databricks-sdk

# Restart Python 
dbutils.library.restartPython()

In [ ]:
# 📚 Import Libraries
import mlflow
import json
import time
import pandas as pd
from mlflow.tracking import MlflowClient
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedModelInput
from pyspark.sql import SparkSession

# Initialize
spark = SparkSession.builder.getOrCreate()
mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

print("✅ Libraries loaded for model deployment")

In [ ]:
# 🎯 Configuration - Get from Bundle Parameters
catalog_name = spark.conf.get("catalog_name", "dev_digital_engineering_services")
schema_name = spark.conf.get("schema_name", "default")
model_name = spark.conf.get("model_name", "water_quality_model")
env = spark.conf.get("env", "dev")

# Construct names
FULL_MODEL_NAME = f"{catalog_name}.{schema_name}.{model_name}"
ENDPOINT_NAME = f"water-quality-{env}"

print(f"🎯 Deployment Configuration:")
print(f"   🤖 Model: {FULL_MODEL_NAME}")
print(f"   🌐 Endpoint: {ENDPOINT_NAME}")
print(f"   🏷️ Environment: {env}")

In [ ]:
# ✅ Verify Challenger Approval
print("✅ Verifying Challenger model approval...")

try:
    # Get the Challenger model version
    challenger_version = client.get_model_version_by_alias(FULL_MODEL_NAME, "Challenger")
    
    print(f"🔍 Found Challenger version: {challenger_version.version}")
    
    # Check deployment ready status
    tags = challenger_version.tags or {}
    deployment_ready = tags.get("deployment_ready", "false")
    approval_status = tags.get("approval", "not_set")
    
    if deployment_ready != "true" or approval_status != "approved":
        print(f"❌ Model not ready for deployment")
        print(f"   Deployment ready: {deployment_ready}")
        print(f"   Approval status: {approval_status}")
        print("💡 Run evaluation and approval workflows first")
        raise Exception("Model not approved for deployment")
    
    print(f"✅ Challenger model verified and approved for deployment")
    
except Exception as e:
    print(f"❌ Error: {e}")
    raise e

In [ ]:
# 🏆 Promote Challenger to Champion
print("🏆 Promoting Challenger to Champion...")

try:
    # Check if Champion alias already exists
    try:
        existing_champion = client.get_model_version_by_alias(FULL_MODEL_NAME, "Champion")
        print(f"ℹ️ Existing Champion version: {existing_champion.version}")
        
        # Archive the old champion
        client.set_model_version_tag(
            name=FULL_MODEL_NAME,
            version=existing_champion.version,
            key="stage",
            value="archived"
        )
        print(f"📦 Previous Champion (v{existing_champion.version}) archived")
        
    except Exception:
        print(f"ℹ️ No existing Champion found")
    
    # Promote Challenger to Champion
    client.set_registered_model_alias(
        name=FULL_MODEL_NAME,
        alias="Champion",
        version=challenger_version.version
    )
    
    # Update tags
    client.set_model_version_tag(
        name=FULL_MODEL_NAME,
        version=challenger_version.version,
        key="stage",
        value="champion"
    )
    
    client.set_model_version_tag(
        name=FULL_MODEL_NAME,
        version=challenger_version.version,
        key="promotion_date",
        value=pd.Timestamp.now().isoformat()
    )
    
    print(f"✅ CHAMPION PROMOTION SUCCESSFUL!")
    print(f"🏆 Version {challenger_version.version} is now Champion")
    
except Exception as e:
    print(f"❌ Error promoting to Champion: {e}")
    raise e

In [ ]:
# 🌐 Create/Update Model Serving Endpoint
print("🌐 Creating Model Serving endpoint...")

# Initialize Databricks SDK
w = WorkspaceClient()

# Check if endpoint already exists
existing_endpoints = w.serving_endpoints.list()
endpoint_exists = any(ep.name == ENDPOINT_NAME for ep in existing_endpoints)

if endpoint_exists:
    print(f"ℹ️ Endpoint '{ENDPOINT_NAME}' already exists - updating...")
    
    # Update existing endpoint
    w.serving_endpoints.update_config_and_wait(
        name=ENDPOINT_NAME,
        served_models=[
            ServedModelInput(
                model_name=FULL_MODEL_NAME,
                model_version=challenger_version.version,
                workload_size="Small",
                scale_to_zero_enabled=True,
                name="water_quality_model_current"
            )
        ]
    )
    print(f"✅ Endpoint updated with Champion model v{challenger_version.version}")
    
else:
    print(f"🚀 Creating new endpoint '{ENDPOINT_NAME}'...")
    
    # Create new endpoint
    w.serving_endpoints.create_and_wait(
        name=ENDPOINT_NAME,
        config=EndpointCoreConfigInput(
            served_models=[
                ServedModelInput(
                    model_name=FULL_MODEL_NAME,
                    model_version=challenger_version.version,
                    workload_size="Small", 
                    scale_to_zero_enabled=True,
                    name="water_quality_model_current"
                )
            ]
        )
    )
    print(f"✅ New endpoint created successfully")

print(f"🌐 Endpoint Name: {ENDPOINT_NAME}")
print(f"🤖 Model: {FULL_MODEL_NAME}")
print(f"📦 Version: {challenger_version.version}")

In [ ]:
# 🧪 Test Endpoint with Sample Data
print("🧪 Testing endpoint with sample water quality data...")

time.sleep(10)  # Wait for endpoint to be ready

# Sample water quality data for testing
test_samples = [
    {
        "ph": 7.2, "Hardness": 180.5, "Solids": 15000, "Chloramines": 8.2,
        "Sulfate": 350, "Conductivity": 400, "Organic_carbon": 12,
        "Trihalomethanes": 70, "Turbidity": 3.5,
        "ph_normalized": 7.2/14, 
        "hardness_ratio": 180.5/(15000+1),
        "organic_load": 12*8.2,
        "water_quality_index": (7.2/14 + 1/(3.5+1) + 1/(180.5/(15000+1)+1))/3
    },
    {
        "ph": 6.8, "Hardness": 220.0, "Solids": 18000, "Chloramines": 7.5,
        "Sulfate": 280, "Conductivity": 380, "Organic_carbon": 14,
        "Trihalomethanes": 85, "Turbidity": 4.2,
        "ph_normalized": 6.8/14,
        "hardness_ratio": 220.0/(18000+1), 
        "organic_load": 14*7.5,
        "water_quality_index": (6.8/14 + 1/(4.2+1) + 1/(220.0/(18000+1)+1))/3
    }
]

try:
    # Test endpoint with sample data
    for i, sample in enumerate(test_samples):
        print(f"\n🧪 Test Sample {i+1}:")
        
        # Format data for endpoint
        data_json = {"dataframe_records": [sample]}
        
        # Call endpoint
        response = w.serving_endpoints.query(
            name=ENDPOINT_NAME,
            dataframe_records=[sample]
        )
        
        prediction = response.predictions[0]
        result = "✅ POTABLE" if prediction == 1 else "❌ NOT POTABLE"
        
        print(f"   📊 Input pH: {sample['ph']}, Turbidity: {sample['Turbidity']}")
        print(f"   🎯 Prediction: {result} (class: {prediction})")
    
    print(f"\n✅ ENDPOINT TESTING SUCCESSFUL!")
    
except Exception as e:
    print(f"⚠️ Endpoint test failed (might still be starting): {e}")
    print(f"💡 Endpoint is created but may need a few minutes to be ready")

In [ ]:
# 📊 Deployment Summary
print("📊 DEPLOYMENT SUMMARY:")
print("=" * 50)
print(f"🏆 Champion Model: {FULL_MODEL_NAME}")
print(f"📦 Version: {challenger_version.version}")
print(f"🌐 Endpoint: {ENDPOINT_NAME}")
print(f"🎯 Environment: {env}")
print(f"⚡ Workload Size: Small")
print(f"🔄 Scale to Zero: Enabled")
print(f"")

# Get endpoint details
try:
    endpoint_details = w.serving_endpoints.get(ENDPOINT_NAME)
    print(f"📊 Endpoint Status: {endpoint_details.state.config_update}")
    print(f"🌐 Endpoint URL: Available in Databricks UI")
except Exception as e:
    print(f"ℹ️ Endpoint details: {e}")

print(f"")
print(f"✅ DEPLOYMENT COMPLETED SUCCESSFULLY!")
print(f"")
print(f"🎯 NEXT STEPS:")
print(f"   1. ✅ Model is live for real-time inference")
print(f"   2. 🔄 Run batch inference job for scheduled predictions")  
print(f"   3. 📊 Set up monitoring for drift detection")
print(f"   4. 🌐 Access endpoint at: Databricks → Machine Learning → Serving")

# Log successful deployment
deployment_info = {
    "model_name": FULL_MODEL_NAME,
    "version": challenger_version.version,
    "endpoint_name": ENDPOINT_NAME,
    "deployment_date": pd.Timestamp.now().isoformat(),
    "environment": env
}

print(f"")
print(f"🚀 WATER QUALITY MODEL DEPLOYED TO PRODUCTION! 🎉")